## 辅助函数

`run_script()` 通过 `subprocess.run` 调用 bash 脚本，支持位置参数和环境变量。

In [1]:
import os
import subprocess

PROJECT_ROOT = os.getcwd()

def run_script(script_path, *args, env=None, cwd=None):
    """Run a shell script with positional args and optional environment variables.

    Parameters
    ----------
    script_path : str
        Path to the bash script.
    *args : str
        Positional arguments passed to the script ($1, $2, ...).
    env : dict, optional
        Environment variables to set for this run (e.g. {'TRAIN_LR': '0.001'}).
        These override the script's defaults without modifying the host environment.
    cwd : str, optional
        Working directory for the subprocess (defaults to PROJECT_ROOT).
    """
    if cwd is None:
        cwd = PROJECT_ROOT
    cmd = ['bash', script_path] + list(args)
    env_str = ' '.join(f'{k}={v}' for k, v in (env or {}).items())
    print(f"Running: {env_str} {' '.join(cmd)}")
    run_env = os.environ.copy()
    if env:
        run_env.update(env)
    result = subprocess.run(cmd, cwd=cwd, env=run_env, text=True)
    if result.returncode != 0:
        print(f"Error: Script exited with code {result.returncode}")
    return result.returncode

## Step 1: 预训练

调用 `pretrain/run.sh`，启动 XPert 模型的大规模预训练。

### 位置参数

| 位置 | 参数名 | 默认值 | 说明 |
|------|--------|--------|------|
| 1 | `dataset` | `l1000_mdmt_full` | 数据集名。可选：`l1000_mdmt_full`（336,852 样本）、`l1000_mdmt_pretrain`（268,022 样本） |
| 2 | `split_method` | `pretrain_split` | 数据划分方式，由 YAML `split_methods` 解析为 obs 列名 |
| 3 | `config` | `pretrain/config/l1000` | YAML 配置文件路径（相对项目根目录，不含 `.yaml`） |

### 训练超参数（环境变量）

| 环境变量 | 默认值 | 说明 |
|----------|--------|------|
| `TRAIN_LR` | `0.004` | 初始学习率 |
| `WEIGHT_DECAY` | `0.00001` | L2 正则化强度 |
| `BATCH_SIZE` | `64` | 每批样本数 |
| `NUM_EPOCHS` | `2500` | 训练总 epoch 数 |
| `INIT_EPOCH` | `70` | Warmup epoch 数（前 N 个 epoch 冻结参数不更新） |
| `LOSS_WEIGHT` | `0.2,0.003,0.2,1` | 四项损失权重：`L_pert, L_ctl, L_deg, L_pcc` |
| `PATIENCE` | `50` | Early stopping 耐心值 |
| `BANDWIDTH` | `0.3` | PCC loss 的 kernel bandwidth |

### 药物嵌入（环境变量）

| 环境变量 | 默认值 | 说明 |
|----------|--------|------|
| `DRUG_EMBEDS` | `unimol,HG` | 使用的嵌入，逗号分隔。可选：`unimol`, `KPGT`, `morgan`, `smi`, `HG` |
| `DRUG_FEAT` | `unimol` | 主分子特征类型。可选：`unimol`, `KPGT`, `morgan`, `smi` |

### 输入文件

- `pretrain/data/l1000_mdmt_full_336852.h5ad` — 预训练 h5ad 数据
- `drug_embeddings/all_drugs_unimol_arr.npy` — UniMol 分子特征（8981 x 122 x 514）
- `drug_embeddings/all_drugs_idx2HG.npy` — HG 药物嵌入（8981 x 256）
- `drug_embeddings/PPI_gene_vector_128d.npy` — PPI 基因向量（978 x 128）

### 输出

- `pretrain/output/{dataset}_{split_method}/{timestamp}/` — 模型权重与训练日志

In [ ]:
# 基础用法：使用所有默认参数
# run_script('pretrain/run.sh')

# 完整用法：显式指定所有参数
run_script('pretrain/run.sh',
    # 位置参数
    'l1000_mdmt_full',            # $1: dataset
    'pretrain_split',              # $2: split_method (YAML 中唯一选项)
    'pretrain/config/l1000',       # $3: config
    # 环境变量（覆盖脚本默认值）
    env={
        # 训练超参数
        'TRAIN_LR':      '0.004',
        'WEIGHT_DECAY':  '0.00001',
        'BATCH_SIZE':    '64',
        'NUM_EPOCHS':    '2500',
        'INIT_EPOCH':    '70',
        'LOSS_WEIGHT':   '0.2,0.003,0.2,1',
        'PATIENCE':      '50',
        'BANDWIDTH':     '0.3',
        # 药物嵌入
        'DRUG_EMBEDS':   'unimol,HG',
        'DRUG_FEAT':     'unimol',
    })


## Step 1b: 预训练（Example 快速测试）

使用小样本数据集 `l1000_mdmt_example`（384 样本）快速验证预训练流程。

与完整预训练的区别：
- **数据量极小**：384 样本（完整数据集的 0.11%），训练很快完成
- **epoch 数少**：`NUM_EPOCHS=10`，仅用于流程验证
- **split_method 相同**：复用 `pretrain_split`
- **用途**：测试代码修改、验证环境配置、快速迭代

### 位置参数

| 位置 | 参数名 | 值 | 说明 |
|------|--------|--------|------|
| 1 | `dataset` | `l1000_mdmt_example` | 小样本数据集 |
| 2 | `split_method` | `pretrain_split` | 同完整预训练 |
| 3 | `config` | `pretrain/config/l1000` | 复用同一份配置文件 |

### 训练超参数（环境变量）

| 环境变量 | 默认值 | Example 用值 | 说明 |
|----------|--------|-------------|------|
| `NUM_EPOCHS` | `2500` | `10` | 大幅减少，快速完成 |
| `BATCH_SIZE` | `64` | `16` | 样本量少，用小 batch |
| `PATIENCE` | `50` | `5` | 快速 early stop |
| 其他 | 见 Step 1 | 保持默认 | 学习率、损失权重等不变 |

### 预期运行时间

约 1-2 分钟（GPU），用于验证代码无报错、数据加载正常。

In [ ]:
# 小样本快速测试：仅 10 epochs，用于验证流程
run_script('pretrain/run.sh',
    # 位置参数
    'l1000_mdmt_example',              # $1: 小样本数据集
    'pretrain_split',                   # $2: split_method
    'pretrain/config/l1000',            # $3: 复用同一配置文件
    env={
        # 训练超参数
        'TRAIN_LR':      '0.004',
        'WEIGHT_DECAY':  '0.00001',
        'BATCH_SIZE':    '16',
        'NUM_EPOCHS':    '10',
        'INIT_EPOCH':    '3',
        'LOSS_WEIGHT':   '0.2,0.003,0.2,1',
        'PATIENCE':      '5',
        'BANDWIDTH':     '0.3',
        # 药物嵌入
        'DRUG_EMBEDS':   'unimol,HG',
        'DRUG_FEAT':     'unimol',
    })


## Step 2: 微调

调用 `finetune/run.sh`，使用预训练模型 `pretrain/output/pretrain_mdmt_full_200_epoch.pth` 分别在 CDSDB、PANACEA、DTOX 三个数据集上微调。

### 位置参数

| 位置 | 参数名 | 默认值 | 说明 |
|------|--------|--------|------|
| 1 | `pretrained_model_path` | **必填** | 预训练权重路径 |
| 2 | `dataset` | `cdsdb` | 目标数据集。可选：`cdsdb`、`panacea`、`dtoxs` |
| 3 | `split_method` | `random` | 数据划分方式：`random`、`warm-drug`、`cold-drug` |
| 4 | `config` | `finetune/config/cdsdb` | YAML 配置文件路径 |

### 划分方式说明

| split_method | 说明 |
|------|------|
| `random` | 随机 7:1.5:1.5 划分，5 fold |
| `warm-drug` | test 中药物均在 train 中出现过，按 8.5:1.5 分 train+valid 和 test，再 5:1 分 train:valid |
| `cold-drug` | test 中药物均未在 train 中出现过，按 8.5:1.5 分 train+valid 和 test，再 5:1 分 train:valid |

### 训练超参数（环境变量）

| 环境变量 | 默认值 | 说明 |
|----------|--------|------|
| `TRAIN_LR` | `0.004` | 初始学习率 |
| `WEIGHT_DECAY` | `0.00001` | L2 正则化强度 |
| `BATCH_SIZE` | `32` | 每批样本数 |
| `NUM_EPOCHS` | `2` | 训练总 epoch 数 |
| `INIT_EPOCH` | `0` | Warmup epoch 数（微调通常为 0） |
| `LOSS_WEIGHT` | `0.2,0.003,0.2,1` | 四项损失权重 |
| `PATIENCE` | `50` | Early stopping 耐心值 |
| `BANDWIDTH` | `0.3` | PCC loss kernel bandwidth |

### 微调超参数（环境变量）

| 环境变量 | 默认值 | 说明 |
|----------|--------|------|
| `FINETUNE_LR` | `0.002` | 微调阶段学习率 |
| `FINETUNE_WEIGHT_DECAY` | `0.00001` | 微调阶段 L2 正则化 |
| `FINETUNE_PARAMS` | `all` | 微调参数范围（`all` 或具体 layer 前缀） |
| `EXCLUDE_KEYS` | （空） | 排除不加载预训练权重的 key 子串，逗号分隔 |

### 药物嵌入（环境变量）

| 环境变量 | 默认值 | 说明 |
|----------|--------|------|
| `DRUG_EMBEDS` | `unimol,HG` | 使用的嵌入，逗号分隔 |
| `DRUG_FEAT` | `unimol` | 主分子特征类型 |

### 输出

`finetune/output/{dataset}_{split_method}/{timestamp}/` — 权重文件命名为 `{dataset}_{split_method}_fold-{n}_best.pth`

In [ ]:
# DTOX 微调
PRETRAINED_MODEL = 'pretrain/output/pretrain_mdmt_full_50_epoch.pth'

run_script('finetune/run.sh',
    PRETRAINED_MODEL,
    'dtoxs',
    'random',
    'finetune/config/dtox',
    env={
        'TRAIN_LR':              '0.004',
        'WEIGHT_DECAY':          '0.00001',
        'BATCH_SIZE':            '32',
        'NUM_EPOCHS':            '2',
        'INIT_EPOCH':            '0',
        'LOSS_WEIGHT':           '0.2,0.003,0.2,1',
        'PATIENCE':              '50',
        'BANDWIDTH':             '0.3',
        'FINETUNE_LR':           '0.002',
        'FINETUNE_WEIGHT_DECAY': '0.00001',
        'FINETUNE_PARAMS':       'all',
        'EXCLUDE_KEYS':          '',
        'DRUG_EMBEDS':           'unimol,HG',
        'DRUG_FEAT':             'unimol',
    })


In [ ]:
# CDSDB 微调
PRETRAINED_MODEL = 'pretrain/output/pretrain_mdmt_full_50_epoch.pth'

run_script('finetune/run.sh',
    PRETRAINED_MODEL,
    'cdsdb',
    'random',
    'finetune/config/cdsdb',
    env={
        'TRAIN_LR':              '0.004',
        'WEIGHT_DECAY':          '0.00001',
        'BATCH_SIZE':            '32',
        'NUM_EPOCHS':            '2',
        'INIT_EPOCH':            '0',
        'LOSS_WEIGHT':           '0.2,0.003,0.2,1',
        'PATIENCE':              '50',
        'BANDWIDTH':             '0.3',
        'FINETUNE_LR':           '0.002',
        'FINETUNE_WEIGHT_DECAY': '0.00001',
        'FINETUNE_PARAMS':       'all',
        'EXCLUDE_KEYS':          '',
        'DRUG_EMBEDS':           'unimol,HG',
        'DRUG_FEAT':             'unimol',
    })


In [ ]:
# PANACEA 微调
PRETRAINED_MODEL = 'pretrain/output/pretrain_mdmt_full_50_epoch.pth'

run_script('finetune/run.sh',
    PRETRAINED_MODEL,
    'panacea',
    'random',
    'finetune/config/panacea',
    env={
        'TRAIN_LR':              '0.004',
        'WEIGHT_DECAY':          '0.00001',
        'BATCH_SIZE':            '32',
        'NUM_EPOCHS':            '2500',
        'INIT_EPOCH':            '500',
        'LOSS_WEIGHT':           '0.2,0.003,0.2,1',
        'PATIENCE':              '50',
        'BANDWIDTH':             '0.3',
        'FINETUNE_LR':           '0.002',
        'FINETUNE_WEIGHT_DECAY': '0.00001',
        'FINETUNE_PARAMS':       'all',
        'EXCLUDE_KEYS':          '',
        'DRUG_EMBEDS':           'unimol,HG',
        'DRUG_FEAT':             'unimol',
    })
